# 🎧 Sonic Atlas
### *Map the hidden geography of your music taste*

This notebook turns your Spotify Liked Songs into an **interactive network of artists**, where edges represent musical similarity. The result is a scrollytelling web dashboard you can explore and share.

---

### What you'll build

```
Spotify Liked Songs
      ↓ OAuth PKCE
  Your Artists (nodes)
      ↓ Last.fm getSimilar
  Similarity Edges (weighted 0–1)
      ↓ NetworkX + Louvain
  Communities (taste tribes)
      ↓ Claude API (optional)
  Tribe labels + personality descriptions
      ↓ HTML template
  sonic_atlas.html  ← interactive dashboard
```

### Estimated runtime
| Step | Time |
|------|------|
| Spotify fetch (2000+ songs) | ~2 min |
| Last.fm similarity (1800+ artists) | ~8–10 min |
| Graph metrics + Louvain | ~2 min |
| Claude API (labels + descriptions) | ~2 min |
| **Total** | **~15–20 min** |

### Prerequisites
- A Spotify account with Liked Songs
- API keys (see **Cell 1** for setup instructions)
- Google Colab or Jupyter Notebook


## 🔑 API Keys Setup

You need **3 API keys** to run this notebook. Follow the steps below to get each one, then store them using the method for your environment.

---

### Where to put your keys

| Environment | Where to store keys |
|-------------|-------------------|
| **Google Colab** | Click the 🔑 **Secrets** icon in the left sidebar → "Add new secret". Use the exact variable names below. Enable "Notebook access" for each. |
| **Local Jupyter** | Edit `sonic-atlas/.env` (already gitignored). The file was created for you with placeholder values. |

---

### 1. Spotify — Client ID & Client Secret

Secret names: `SPOTIFY_CLIENT_ID`, `SPOTIFY_CLIENT_SECRET`

1. Go to **[developer.spotify.com/dashboard](https://developer.spotify.com/dashboard)**
2. Log in → click **"Create app"** and fill in:
   - App name: `sonic-atlas`
   - Redirect URI: `http://127.0.0.1:8888/callback` → click **Add**
   - Check **Web API** → accept the terms → **Save**
3. Open the app → **Settings** → copy **Client ID** and **Client Secret**
4. Go to **User Management** → add your Spotify email address

> ⚠️ The redirect URI `http://127.0.0.1:8888/callback` must be added exactly as shown.

---

### 2. Last.fm — API Key

Secret name: `LASTFM_API_KEY`

> **Why Last.fm and not Spotify?** Spotify's `/artists/{id}/related-artists` and `/audio-features` endpoints were removed for apps in Development Mode in November 2024. Last.fm offers equivalent (and arguably richer) similarity data, built from the listening patterns of millions of users worldwide — and it's completely free.

1. Go to **[last.fm/api/account/create](https://www.last.fm/api/account/create)**
2. Fill in:
   - Application name: `sonic-atlas`
   - Application description: `Music network analysis`
3. Submit → your **API Key** appears immediately

---

### 3. Anthropic — API Key *(optional)*

Secret name: `ANTHROPIC_API_KEY`

> If you leave this blank, the notebook will generate automatic labels from the top artists in each community.

1. Go to **[console.anthropic.com](https://console.anthropic.com)**
2. **Settings → API Keys → Create Key**
3. Add billing credits at **Settings → Billing** (minimum $5)
   - This project costs approximately **$0.10–0.15** total

---

Once you have all keys, run **Cell 3 (Configuration)** — it will load them automatically.


In [1]:
# ════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════
# Keys are loaded from the right place automatically:
#   • Google Colab  → Secrets panel (🔑 left sidebar) — no file needed
#   • Local Jupyter → sonic-atlas/.env             — gitignored, never committed
# ─────────────────────────────────────────────────────────────────────────────
import os

# ── Detect runtime ────────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── Load secrets ──────────────────────────────────────────────────────────────
if IN_COLAB:
    def _secret(key, default=""):
        try:
            return userdata.get(key) or default
        except Exception:
            return default
    print("🌐 Running in Google Colab — reading from Secrets panel.")
else:
    from dotenv import load_dotenv
    load_dotenv()   # searches cwd and parent dirs for .env
    def _secret(key, default=""):
        return os.getenv(key, default)
    print("💻 Running locally — reading from .env file.")

# ── Spotify credentials ───────────────────────────────────────────────────────
SPOTIFY_CLIENT_ID     = _secret("SPOTIFY_CLIENT_ID")
SPOTIFY_CLIENT_SECRET = _secret("SPOTIFY_CLIENT_SECRET")
SPOTIFY_REDIRECT_URI  = _secret("SPOTIFY_REDIRECT_URI", "http://127.0.0.1:8888/callback")
SPOTIFY_SCOPE         = "user-library-read"

# ── Last.fm credentials ───────────────────────────────────────────────────────
LASTFM_API_KEY = _secret("LASTFM_API_KEY")

# ── Anthropic credentials (optional) ─────────────────────────────────────────
ANTHROPIC_API_KEY = _secret("ANTHROPIC_API_KEY")
ANTHROPIC_MODEL   = "claude-sonnet-4-6"

# ── HTML template (only needed in the last cell — Step 9) ─────────────────────
# After uploading sonic_atlas_template.html to a GitHub Gist,
# paste the Raw URL here or add TEMPLATE_URL to your .env / Colab Secrets.
TEMPLATE_URL = _secret("TEMPLATE_URL")

# ── GitHub (only needed in Step 10 — push to GitHub Pages) ───────────────────
# Add GITHUB_USERNAME and GITHUB_REPO to your .env / Colab Secrets.
GITHUB_USERNAME = _secret("GITHUB_USERNAME")
GITHUB_REPO     = _secret("GITHUB_REPO")

# ── Visualisation parameters ──────────────────────────────────────────────────
TOP_N_COMMUNITIES    = 10    # Number of taste tribes to show in the dashboard
TOP_N_NODES          = 1883  # Max nodes in visualisation (reduce if browser is slow)
MIN_EDGE_WEIGHT      = 0.4   # Minimum Last.fm similarity score to include an edge
OBSESSION_THRESHOLD  = 6     # Min tracks saved to show floating label in Act 4
TOP_N_DISCOVERIES    = 50    # Number of discovery candidates to include

# ── Validate required fields ──────────────────────────────────────────────────
errors   = []
warnings = []
if not SPOTIFY_CLIENT_ID:     errors.append("SPOTIFY_CLIENT_ID is missing")
if not SPOTIFY_CLIENT_SECRET: errors.append("SPOTIFY_CLIENT_SECRET is missing")
if not LASTFM_API_KEY:        errors.append("LASTFM_API_KEY is missing")
if not GITHUB_USERNAME:       warnings.append("GITHUB_USERNAME is empty — needed only in Step 10 (push to GitHub Pages)")
if not GITHUB_REPO:           warnings.append("GITHUB_REPO is empty — needed only in Step 10 (push to GitHub Pages)")

if errors:
    src = "Colab Secrets (🔑 sidebar)" if IN_COLAB else "sonic-atlas/.env"
    print("❌ Missing required configuration:")
    for e in errors:
        print(f"   · {e}")
    print(f"\nAdd the missing keys to {src} and re-run this cell.")
else:
    print("✅ Configuration looks good — ready to run!")
    if not ANTHROPIC_API_KEY:
        warnings.append("ANTHROPIC_API_KEY is empty — tribe labels will be auto-generated")
    for w in warnings:
        print(f"ℹ️  {w}")


💻 Running locally — reading from .env file.
✅ Configuration looks good — ready to run!


In [2]:
# ════════════════════════════════════════════════════════════════════════════
# INSTALL DEPENDENCIES
# %pip ensures packages are installed into the active kernel (works in both
# Colab and local Jupyter — unlike !pip which can target the wrong Python)
# ════════════════════════════════════════════════════════════════════════════
%pip install requests networkx tqdm numpy python-louvain python-dotenv ipywidgets --quiet
print("✅ All dependencies installed.")


Note: you may need to restart the kernel to use updated packages.
✅ All dependencies installed.


In [3]:
# ════════════════════════════════════════════════════════════════════════════
# IMPORTS & HELPER FUNCTIONS
# ════════════════════════════════════════════════════════════════════════════
import base64, hashlib, os, re, json, time, unicodedata
import urllib.parse as urlparse
import requests
import numpy as np
import networkx as nx
from collections import defaultdict, Counter
from itertools import combinations
from tqdm.auto import tqdm   # auto: widget bar in Jupyter, text bar as fallback

# ── Spotify auth helpers ──────────────────────────────────────────────────────
ACCESS_TOKEN  = None
REFRESH_TOKEN = None
TOKEN_EXPIRY  = 0

def get_headers():
    global ACCESS_TOKEN, REFRESH_TOKEN, TOKEN_EXPIRY
    if REFRESH_TOKEN is None:
        raise RuntimeError(
            "Not authenticated. Run the Spotify auth cells (Steps 1a & 1b) first.\n"
            "Auth codes are single-use — you need a fresh one after every kernel restart."
        )
    if time.time() > TOKEN_EXPIRY - 60:
        r = requests.post(
            "https://accounts.spotify.com/api/token",
            data={"grant_type": "refresh_token", "refresh_token": REFRESH_TOKEN,
                  "client_id": SPOTIFY_CLIENT_ID},
            headers={"Content-Type": "application/x-www-form-urlencoded"}
        )
        data = r.json()
        if "access_token" not in data:
            raise RuntimeError(
                f"Token refresh failed: {data.get('error_description', data)}\n"
                "Re-run the Spotify auth cells to get a new token."
            )
        ACCESS_TOKEN = data["access_token"]
        TOKEN_EXPIRY = time.time() + data.get("expires_in", 3600)
        print("🔄 Token refreshed.")
    return {"Authorization": f"Bearer {ACCESS_TOKEN}"}

def spotify_get(url, params=None, retries=3):
    for attempt in range(retries):
        r = requests.get(url, headers=get_headers(), params=params)
        if r.status_code == 200:   return r.json()
        elif r.status_code == 429:
            wait = int(r.headers.get("Retry-After", 5))
            print(f"⏳ Rate limited. Waiting {wait}s...")
            time.sleep(wait)
        elif r.status_code == 401:
            get_headers()
        else:
            print(f"⚠️  HTTP {r.status_code}: {r.text[:200]}")
            return None
    return None

# ── General helpers ───────────────────────────────────────────────────────────
def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def normalize_name(name):
    """Lowercase, remove accents and punctuation for fuzzy matching."""
    name = name.lower().strip()
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = re.sub(r"[^\w\s]", "", name)
    return re.sub(r"\s+", " ", name).strip()

def anthropic_call(prompt, max_tokens=200):
    """Call Claude API. Returns text or None if no key configured."""
    if not ANTHROPIC_API_KEY:
        return None
    try:
        r = requests.post(
            "https://api.anthropic.com/v1/messages",
            headers={"x-api-key": ANTHROPIC_API_KEY,
                     "anthropic-version": "2023-06-01",
                     "content-type": "application/json"},
            json={"model": ANTHROPIC_MODEL, "max_tokens": max_tokens,
                  "messages": [{"role": "user", "content": prompt}]},
            timeout=30
        )
        return r.json()["content"][0]["text"].strip()
    except Exception as e:
        print(f"⚠️  Anthropic API error: {e}")
        return None

print("✅ Helpers loaded.")


✅ Helpers loaded.


In [4]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 1 — SPOTIFY AUTHENTICATION (OAuth PKCE)
# ════════════════════════════════════════════════════════════════════════════
#
# How it works:
#   1. Run this cell → it prints an authorization URL
#   2. Open that URL in your browser and log in to Spotify
#   3. You'll be redirected to a page that says "This site can't be reached"
#      — that's expected! Copy the full URL from the browser address bar
#   4. Run the next cell and paste that URL when prompted
# ─────────────────────────────────────────────────────────────────────────────

code_verifier = base64.urlsafe_b64encode(os.urandom(32)).rstrip(b"=").decode()
code_challenge = base64.urlsafe_b64encode(
    hashlib.sha256(code_verifier.encode()).digest()
).rstrip(b"=").decode()

auth_url = "https://accounts.spotify.com/authorize?" + urlparse.urlencode({
    "client_id":             SPOTIFY_CLIENT_ID,
    "response_type":         "code",
    "redirect_uri":          SPOTIFY_REDIRECT_URI,
    "scope":                 SPOTIFY_SCOPE,
    "code_challenge_method": "S256",
    "code_challenge":        code_challenge,
})

print("=" * 60)
print("SONIC ATLAS — Spotify Authentication")
print("=" * 60)
print()
print("Step 1: Open this URL in your browser:")
print()
print(auth_url)
print()
print("Step 2: Log in and authorise access.")
print("Step 3: The browser will redirect to a page that won't load.")
print("        Copy the full URL from the address bar.")
print("Step 4: Run the next cell and paste that URL.")


SONIC ATLAS — Spotify Authentication

Step 1: Open this URL in your browser:

https://accounts.spotify.com/authorize?client_id=7f370ba1a4974751b0f042c7992820d1&response_type=code&redirect_uri=http%3A%2F%2F127.0.0.1%3A8888%2Fcallback&scope=user-library-read&code_challenge_method=S256&code_challenge=DXL_gGQGhBQ4CRBGN9bJbnxzde1gO3D5dhdfURbLKjU

Step 2: Log in and authorise access.
Step 3: The browser will redirect to a page that won't load.
        Copy the full URL from the address bar.
Step 4: Run the next cell and paste that URL.


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 1b — PASTE THE REDIRECT URL HERE
# ════════════════════════════════════════════════════════════════════════════
#
# Paste the full redirect URL below (starts with http://127.0.0.1:8888/callback?code=...)
# ⚠️  Auth codes are single-use and expire in ~60 seconds.
#     You need a fresh one after every kernel restart.
# ─────────────────────────────────────────────────────────────────────────────

redirect_url = ""  # ← paste the redirect URL here (get a new one from Step 1a)

# ── Extract auth code ─────────────────────────────────────────────────────────
if not redirect_url:
    print("❌ Please paste the redirect URL above and re-run this cell.")
else:
    match = re.search(r"[?&]code=([^&\s]+)", redirect_url)
    auth_code = match.group(1) if match else None

    if not auth_code:
        print("❌ Could not extract auth code. Make sure you pasted the full URL.")
    else:
        r = requests.post(
            "https://accounts.spotify.com/api/token",
            data={"grant_type":    "authorization_code",
                  "code":          auth_code,
                  "redirect_uri":  SPOTIFY_REDIRECT_URI,
                  "client_id":     SPOTIFY_CLIENT_ID,
                  "code_verifier": code_verifier},
            headers={"Content-Type": "application/x-www-form-urlencoded"}
        )
        data = r.json()
        ACCESS_TOKEN  = data.get("access_token")
        REFRESH_TOKEN = data.get("refresh_token")
        TOKEN_EXPIRY  = time.time() + data.get("expires_in", 3600)

        if ACCESS_TOKEN:
            print("✅ Authentication successful! You can now run the next cells.")
        else:
            print("❌ Authentication failed:", data.get("error_description", data))
            print("   The auth code may have expired (valid for ~60 seconds).")
            print("   Re-run Step 1a to generate a new URL and try again.")


✅ Authentication successful! You can now run the next cells.


In [6]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 2 — FETCH LIKED SONGS
# ════════════════════════════════════════════════════════════════════════════
print("📚 Fetching Liked Songs...")

tracks = []
first_page = spotify_get("https://api.spotify.com/v1/me/tracks", {"limit": 50})
if not first_page:
    print("❌ Failed to fetch tracks. Check your token.")
else:
    total = first_page["total"]
    print(f"   Total songs in library: {total}")
    tracks.extend(first_page["items"])

    with tqdm(total=total, initial=len(tracks), desc="Songs") as pbar:
        while first_page.get("next"):
            first_page = spotify_get(first_page["next"])
            if first_page:
                tracks.extend(first_page["items"])
                pbar.update(len(first_page["items"]))
            else:
                break

    print(f"✅ {len(tracks)} songs loaded.")


📚 Fetching Liked Songs...
   Total songs in library: 2324


Songs:   2%|2         | 50/2324 [00:00<?, ?it/s]

✅ 2324 songs loaded.


In [7]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 3 — EXTRACT UNIQUE ARTISTS
# ════════════════════════════════════════════════════════════════════════════
print("🎤 Extracting unique artists...")

artist_track_count = {}   # artist_id → number of saved tracks
artists_data = {}         # artist_id → artist dict

for item in tqdm(tracks, desc="Processing tracks"):
    track = item.get("track")
    if not track:
        continue
    for artist in track.get("artists", []):
        aid = artist.get("id")
        if not aid:
            continue
        if aid not in artists_data:
            artists_data[aid] = {
                "id":          aid,
                "name":        artist["name"],
                "track_count": 0,
                "degree":      0,
                "betweenness": 0.0,
                "community":   -1,
            }
        artists_data[aid]["track_count"] += 1

print(f"✅ {len(artists_data)} unique artists found.")
print(f"   Top 10 by track count:")
top = sorted(artists_data.values(), key=lambda x: x["track_count"], reverse=True)[:10]
for a in top:
    print(f"   · {a['name']}: {a['track_count']} tracks")


🎤 Extracting unique artists...


Processing tracks:   0%|          | 0/2324 [00:00<?, ?it/s]

✅ 1884 unique artists found.
   Top 10 by track count:
   · Robert Glasper: 25 tracks
   · Andrew Bird: 24 tracks
   · Sampha: 22 tracks
   · Ólafur Arnalds: 22 tracks
   · Jordan Rakei: 19 tracks
   · Hania Rani: 19 tracks
   · Brad Mehldau: 16 tracks
   · Tigran Hamasyan: 16 tracks
   · Dustin O'Halloran: 16 tracks
   · MARO: 14 tracks


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 4 — LAST.FM NAME MATCHING
# ════════════════════════════════════════════════════════════════════════════
# Spotify and Last.fm sometimes use different artist names (e.g. accents,
# collaborations). This step verifies coverage and builds a correction map.
# Uses a thread pool with a global rate limiter to stay under Last.fm's 5 req/s
# limit — avoiding the rate-limit retries that kill throughput with many workers.
# Network errors are tracked by artist name and retried in a second pass.
# ─────────────────────────────────────────────────────────────────────────────
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── Global rate limiter (also used by Step 5) ────────────────────────────────
_lastfm_lock        = threading.Lock()
_lastfm_next_call   = [0.0]
LASTFM_MIN_INTERVAL = 0.22      # seconds between requests → ~4.5 req/s

def _lastfm_get(params):
    """Rate-limited GET to Last.fm (max ~4.5 req/s, shared across all workers)."""
    with _lastfm_lock:
        now = time.time()
        if now < _lastfm_next_call[0]:
            time.sleep(_lastfm_next_call[0] - now)
        _lastfm_next_call[0] = time.time() + LASTFM_MIN_INTERVAL
    return requests.get("https://ws.audioscrobbler.com/2.0/", params=params, timeout=10)

LASTFM_WORKERS = 10

# Exceptions that signal a transient network problem (worth retrying separately)
_NETWORK_EXC = (
    requests.exceptions.ConnectionError,
    requests.exceptions.Timeout,
    requests.exceptions.ReadTimeout,
    requests.exceptions.ConnectTimeout,
)

def search_lastfm_name(artist_name, retries=3):
    """Find the canonical Last.fm name for a Spotify artist name.

    Returns (lastfm_name, match_type, error_detail) where match_type is:
      'exact'         — normalised names match
      'fuzzy'         — top result used (names differ)
      'network_error' — all retries failed due to network issues
      None            — artist genuinely not found, or unrecoverable API error
    """
    last_net_exc = None
    for attempt in range(retries):
        try:
            r = _lastfm_get(
                {"method": "artist.search", "artist": artist_name,
                 "api_key": LASTFM_API_KEY, "format": "json", "limit": 5})
            data = r.json()

            if "error" in data:
                err_code = data["error"]
                err_msg  = data.get("message", "")
                if err_code == 29 and attempt < retries - 1:
                    time.sleep(2 ** attempt)   # rate-limit backoff: 1s, 2s, …
                    continue
                return None, None, f"lastfm_error_{err_code}: {err_msg}"

            results = data.get("results", {}).get("artistmatches", {}).get("artist", [])
            if not results:
                return None, None, None   # genuinely absent from Last.fm

            norm_query = normalize_name(artist_name)
            for res in results:
                if normalize_name(res["name"]) == norm_query:
                    return res["name"], "exact", None
            return results[0]["name"], "fuzzy", None

        except _NETWORK_EXC as e:
            last_net_exc = e
            if attempt < retries - 1:
                time.sleep(2 ** attempt)   # exponential backoff: 1s, 2s, …
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(1)
            else:
                return None, None, f"other_exception: {type(e).__name__}: {e}"

    # All retries exhausted with network errors
    return None, "network_error", f"network: {type(last_net_exc).__name__}"


def _run_matching_pass(names_list, desc, retries=3):
    """Run one thread-pool pass of name matching over names_list.

    Live tqdm postfix shows: matched / net_err / no_match counts as it runs.
    Returns (corrections, exact_c, fuzzy_c, no_match_c, network_failed, errors_by_type).
    """
    corrections    = {}
    exact_c = fuzzy_c = no_match_c = 0
    network_failed = []
    errors_by_type = {}

    with ThreadPoolExecutor(max_workers=LASTFM_WORKERS) as executor:
        futures = {
            executor.submit(search_lastfm_name, name, retries): name
            for name in names_list
        }
        with tqdm(total=len(futures), desc=desc) as pbar:
            for future in as_completed(futures):
                spotify_name = futures[future]
                lastfm_name, match_type, err = future.result()

                if match_type == "exact":
                    corrections[spotify_name] = lastfm_name
                    exact_c += 1
                elif match_type == "fuzzy":
                    corrections[spotify_name] = lastfm_name
                    fuzzy_c += 1
                elif match_type == "network_error":
                    network_failed.append(spotify_name)
                else:
                    # None: genuinely absent or API error
                    no_match_c += 1
                    if err:
                        err_key = err.split(":")[0]
                        errors_by_type[err_key] = errors_by_type.get(err_key, 0) + 1

                pbar.set_postfix(
                    matched=exact_c + fuzzy_c,
                    net_err=len(network_failed),
                    no_match=no_match_c,
                    refresh=False,
                )
                pbar.update(1)

    return corrections, exact_c, fuzzy_c, no_match_c, network_failed, errors_by_type


# ── Pass 1: main sweep ────────────────────────────────────────────────────────
est_min = int(len(artists_data) * LASTFM_MIN_INTERVAL / 60) + 2
print(f"🔍 Checking Last.fm coverage for {len(artists_data)} artists...")
print(f"   ({LASTFM_WORKERS} workers · 4.5 req/s rate limiter · estimated ~{est_min} min)")

artist_names = [a["name"] for a in artists_data.values()]

name_corrections, exact_count, fuzzy_count, not_found_count, network_failed, errors_by_type = \
    _run_matching_pass(artist_names, desc="Pass 1 — name matching")

# ── Pass 2: retry network failures ───────────────────────────────────────────
if network_failed:
    print(f"\n🔁 Retrying {len(network_failed)} artists that hit network errors "
          f"(5 retries, exponential backoff)...")
    corr2, e2, f2, nf2, still_failed, errs2 = \
        _run_matching_pass(network_failed, desc="Pass 2 — network retry", retries=5)
    name_corrections.update(corr2)
    exact_count     += e2
    fuzzy_count     += f2
    not_found_count += nf2
    for k, v in errs2.items():
        errors_by_type[k] = errors_by_type.get(k, 0) + v
    network_failed = still_failed

# ── Final summary ─────────────────────────────────────────────────────────────
total_found = exact_count + fuzzy_count
coverage    = total_found / len(artists_data) * 100

print(f"\n📊 Coverage results:")
print(f"   ✅ Exact match:      {exact_count}")
print(f"   🔶 Fuzzy match:      {fuzzy_count}")
print(f"   ❌ Not found:        {not_found_count}  ← genuinely absent from Last.fm (or API error)")
print(f"   🌐 Network failures: {len(network_failed)}  ← Spotify name used as fallback in Step 5")
print(f"   Coverage: {coverage:.1f}%  ({total_found}/{len(artists_data)} matched)")

if network_failed:
    sample = network_failed[:10]
    print(f"\n   Artists still unreachable after retry (first {len(sample)}):")
    for name in sample:
        print(f"   · {name}")
    if len(network_failed) > 10:
        print(f"   ... and {len(network_failed) - 10} more")

if errors_by_type:
    print(f"\n⚠️  API / other errors by type:")
    for err, count in sorted(errors_by_type.items(), key=lambda x: -x[1]):
        print(f"   · {err}: {count}×")
    if any("29" in k for k in errors_by_type):
        print("   → Rate limiting still hit. Try increasing LASTFM_MIN_INTERVAL to 0.3.")
else:
    print("   No API/other errors.")

if coverage < 50:
    print(f"\n   ⚠️  Coverage below 50%. If no errors above, these artists likely")
    print(f"   aren't indexed on Last.fm (common for very obscure/local acts).")


🔍 Checking Last.fm coverage for 1884 artists...
   (Using 10 workers + 4.5 req/s rate limiter — estimated ~8 min)


Matching names:   0%|          | 0/1884 [00:00<?, ?it/s]


📊 Coverage results:
   ✅ Exact match:  1317 artists
   🔶 Fuzzy match:  30 artists
   ❌ Not found:    537 artists
   Coverage: 71.5%

⚠️  Errors during fetching:
   · exception: ConnectionError: HTTPSConnectionPool(host='ws.audioscrobbler.com', port=443): Max retries exceeded with url: /2.0/?method=artist.search&artist=Fulton+Lee&api_key=8b1e5a7b844d70af7b1fef10364274d3&format=json&limit=5 (Caused by NewConnectionError("HTTPSConnection(host='ws.audioscrobbler.com', port=443): Failed to establish a new connection: [Errno 101] Network is unreachable")): 1×
   · exception: ConnectionError: HTTPSConnectionPool(host='ws.audioscrobbler.com', port=443): Max retries exceeded with url: /2.0/?method=artist.search&artist=Isaac+Cobb&api_key=8b1e5a7b844d70af7b1fef10364274d3&format=json&limit=5 (Caused by NewConnectionError("HTTPSConnection(host='ws.audioscrobbler.com', port=443): Failed to establish a new connection: [Errno 101] Network is unreachable")): 1×
   · exception: ConnectionError: HTTPSCo

In [9]:
# Check if Moussa Cissoko (or any specific artist) made it into name_corrections
print("In corrections:", "Moussa Cissoko" in name_corrections)
print(f"\nTotal in name_corrections: {len(name_corrections)}")

# How many artists from artists_data are missing corrections
missing = [a["name"] for a in artists_data.values() if a["name"] not in name_corrections]
print(f"Artists without corrections: {len(missing)}")

In corrections: False

Total in name_corrections: 1340
Artists without corrections: 535


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 5 — LAST.FM SIMILARITY EDGES + DISCOVERY CANDIDATES
# ════════════════════════════════════════════════════════════════════════════
# For each artist in your library, this step fetches their most similar
# artists from Last.fm. Artists already in your library become EDGES.
# Artists NOT in your library become DISCOVERY CANDIDATES.
# Uses the same global rate limiter defined in Step 4 (_lastfm_get).
# Network errors are tracked by artist and retried in a second pass.
# ─────────────────────────────────────────────────────────────────────────────

est_min = int(len(artists_data) * LASTFM_MIN_INTERVAL / 60) + 2
print("⏳ Step 5 makes one Last.fm API call per artist in your library.")
print(f"   With {LASTFM_WORKERS} workers + 4.5 req/s rate limiter — estimated ~{est_min} min for ~{len(artists_data)} artists.")
print("   Grab a coffee ☕ — this is the slowest step.\n")

def get_similar_lastfm(artist_name, limit=50, retries=3):
    """Fetch similar artists from Last.fm.

    Returns (similar_list, error_type, error_detail) where error_type is:
      None      — success (similar_list may be empty if artist has no data)
      'network' — transient network failure, worth retrying
      'api'     — Last.fm API error (artist not found, rate limit, etc.)
      'other'   — unexpected exception
    """
    last_net_exc = None
    for attempt in range(retries):
        try:
            r = _lastfm_get(
                {"method": "artist.getSimilar", "artist": artist_name,
                 "api_key": LASTFM_API_KEY, "format": "json", "limit": limit})
            data = r.json()

            if "error" in data:
                err_code = data["error"]
                err_msg  = data.get("message", "")
                if err_code == 29 and attempt < retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                return [], "api", f"lastfm_error_{err_code}: {err_msg}"

            similar = data.get("similarartists", {}).get("artist", [])
            return [(a["name"], float(a["match"])) for a in similar], None, None

        except _NETWORK_EXC as e:
            last_net_exc = e
            if attempt < retries - 1:
                time.sleep(2 ** attempt)   # exponential backoff: 1s, 2s, …
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(1)
            else:
                return [], "other", f"other_exception: {type(e).__name__}: {e}"

    return [], "network", f"network: {type(last_net_exc).__name__}"


# Build normalised name → id lookup
lastfm_to_id = {}
for aid, artist in artists_data.items():
    spotify_name = artist["name"]
    lastfm_name  = name_corrections.get(spotify_name, spotify_name)
    lastfm_to_id[normalize_name(lastfm_name)] = aid
    lastfm_to_id[normalize_name(spotify_name)] = aid

playlist_names = set(lastfm_to_id.keys())

print(f"🎵 Building similarity edges for {len(artists_data)} artists...")

edges_raw            = defaultdict(float)   # (id_a, id_b) → max similarity
discovery_raw        = {}                   # norm_name → candidate dict
errors_by_type       = {}                   # error_key → count
network_failed_items = []                   # items (aid, artist) that hit network errors

artist_items = list(artists_data.items())


def _fetch(item, retries=3):
    aid, artist = item
    spotify_name = artist["name"]
    lastfm_name  = name_corrections.get(spotify_name, spotify_name)
    similar, error_type, err = get_similar_lastfm(lastfm_name, retries=retries)
    return aid, spotify_name, similar, error_type, err


def _process_results(aid, spotify_name, similar):
    """Merge a similarity result into edges_raw and discovery_raw."""
    for sim_name, score in similar:
        sim_norm = normalize_name(sim_name)
        if sim_norm in playlist_names:
            sim_id = lastfm_to_id[sim_norm]
            if sim_id != aid:
                key = tuple(sorted([aid, sim_id]))
                edges_raw[key] = max(edges_raw[key], score)
        else:
            if sim_norm not in discovery_raw:
                discovery_raw[sim_norm] = {
                    "name":         sim_name,
                    "score_sum":    0.0,
                    "count":        0,
                    "max_score":    0.0,
                    "recommenders": [],
                }
            d = discovery_raw[sim_norm]
            d["score_sum"] += score
            d["count"]     += 1
            d["max_score"]  = max(d["max_score"], score)
            if spotify_name not in d["recommenders"] and len(d["recommenders"]) < 10:
                d["recommenders"].append(spotify_name)


# ── Pass 1: main sweep ────────────────────────────────────────────────────────
with ThreadPoolExecutor(max_workers=LASTFM_WORKERS) as executor:
    futures = {executor.submit(_fetch, item): item for item in artist_items}
    with tqdm(total=len(futures), desc="Pass 1 — getSimilar") as pbar:
        for future in as_completed(futures):
            original_item = futures[future]
            aid, spotify_name, similar, error_type, err = future.result()

            if error_type == "network":
                network_failed_items.append(original_item)
            elif error_type:
                key = err.split(":")[0] if err else error_type
                errors_by_type[key] = errors_by_type.get(key, 0) + 1

            _process_results(aid, spotify_name, similar)

            pbar.set_postfix(
                edges=len(edges_raw),
                net_err=len(network_failed_items),
                api_err=sum(errors_by_type.values()),
                refresh=False,
            )
            pbar.update(1)

# ── Pass 2: retry network failures ───────────────────────────────────────────
if network_failed_items:
    print(f"\n🔁 Retrying {len(network_failed_items)} artists that hit network errors "
          f"(5 retries, exponential backoff)...")
    still_failed = []
    with ThreadPoolExecutor(max_workers=LASTFM_WORKERS) as executor:
        futures2 = {
            executor.submit(_fetch, item, 5): item
            for item in network_failed_items
        }
        with tqdm(total=len(futures2), desc="Pass 2 — network retry") as pbar:
            for future in as_completed(futures2):
                original_item = futures2[future]
                aid, spotify_name, similar, error_type, err = future.result()

                if error_type == "network":
                    still_failed.append(original_item)
                elif error_type:
                    key = err.split(":")[0] if err else error_type
                    errors_by_type[key] = errors_by_type.get(key, 0) + 1

                _process_results(aid, spotify_name, similar)

                pbar.set_postfix(
                    edges=len(edges_raw),
                    still_failing=len(still_failed),
                    refresh=False,
                )
                pbar.update(1)
    network_failed_items = still_failed

# ── Build edges list ──────────────────────────────────────────────────────────
edges = [
    {"source": k[0], "target": k[1], "weight": round(v, 4)}
    for k, v in edges_raw.items() if v > 0
]

print(f"\n✅ {len(edges)} similarity edges built.")
print(f"   {len(discovery_raw)} potential discovery candidates collected.")
print(f"   🌐 Network failures: {len(network_failed_items)}  ← no similar artists fetched for these")

weights = [e["weight"] for e in edges]
if weights:
    print(f"   Edge weight — mean: {np.mean(weights):.3f}, max: {max(weights):.3f}")

if errors_by_type:
    print(f"\n⚠️  API / other errors by type:")
    for err, count in sorted(errors_by_type.items(), key=lambda x: -x[1]):
        print(f"   · {err}: {count}×")
    if any("29" in k for k in errors_by_type):
        print("   → Persistent rate limiting. Try increasing LASTFM_MIN_INTERVAL to 0.3.")
else:
    print("   No API/other errors.")

if network_failed_items:
    sample = [a["name"] for _, a in network_failed_items[:10]]
    print(f"\n   Artists still unreachable after retry (first {len(sample)}):")
    for name in sample:
        print(f"   · {name}")
    if len(network_failed_items) > 10:
        print(f"   ... and {len(network_failed_items) - 10} more")


⏳ Step 5 makes one Last.fm API call per artist in your library.
   With 10 workers + 4.5 req/s rate limiter — estimated ~8 min for ~1884 artists.
   Grab a coffee ☕ — this is the slowest step.

🎵 Building similarity edges for 1884 artists...


Last.fm similar:   0%|          | 0/1884 [00:00<?, ?it/s]


✅ 10929 similarity edges built.
   22749 potential discovery candidates collected.
   Edge weight — mean: 0.395, max: 1.000

⚠️  Errors during fetching:
   · lastfm_error_6: The artist you supplied could not be found: 3×
   · exception: ConnectionError: HTTPSConnectionPool(host='ws.audioscrobbler.com', port=443): Max retries exceeded with url: /2.0/?method=artist.getSimilar&artist=Mononeon&api_key=8b1e5a7b844d70af7b1fef10364274d3&format=json&limit=50 (Caused by NewConnectionError("HTTPSConnection(host='ws.audioscrobbler.com', port=443): Failed to establish a new connection: [Errno 101] Network is unreachable")): 2×
   · exception: ConnectionError: HTTPSConnectionPool(host='ws.audioscrobbler.com', port=443): Max retries exceeded with url: /2.0/?method=artist.getSimilar&artist=Samuel+%C3%9Aria&api_key=8b1e5a7b844d70af7b1fef10364274d3&format=json&limit=50 (Caused by NewConnectionError("HTTPSConnection(host='ws.audioscrobbler.com', port=443): Failed to establish a new connection: [Errno 1

In [11]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 6 — GRAPH METRICS & COMMUNITY DETECTION
# ════════════════════════════════════════════════════════════════════════════
import subprocess, sys

# Install python-louvain if needed
subprocess.run(["pip", "install", "python-louvain", "--quiet"], capture_output=True)
for mod in list(sys.modules.keys()):
    if "community" in mod:
        del sys.modules[mod]
from community import community_louvain

print("📐 Building graph and computing metrics...")

G = nx.Graph()
for aid in artists_data:
    G.add_node(aid)
for edge in edges:
    G.add_edge(edge["source"], edge["target"], weight=edge["weight"])

# Betweenness centrality
print("   Computing betweenness centrality...")
betweenness = nx.betweenness_centrality(G, normalized=True, weight="weight")
degree      = dict(G.degree())

for aid in artists_data:
    artists_data[aid]["degree"]      = degree.get(aid, 0)
    artists_data[aid]["betweenness"] = round(betweenness.get(aid, 0.0), 6)

# Louvain community detection
print("   Running Louvain community detection...")
partition = community_louvain.best_partition(G, weight="weight", random_state=42)
for aid in artists_data:
    artists_data[aid]["community"] = partition.get(aid, -1)

# Graph-level stats
comm_sizes = Counter(partition.values())
components = list(nx.connected_components(G))
largest_cc = max(components, key=len)

graph_stats = {
    "n_nodes":                G.number_of_nodes(),
    "n_edges":                G.number_of_edges(),
    "n_components":           len(components),
    "largest_component_size": len(largest_cc),
    "density":                round(nx.density(G), 4),
    "most_connected_artist":  max(degree, key=degree.get),
    "most_isolated_artist":   min(degree, key=degree.get),
    "top_connector":          max(betweenness, key=betweenness.get),
    "n_communities":          len(comm_sizes),
}

nodes_by_id = {n["id"]: n for n in artists_data.values()}
print(f"\n📊 Graph summary:")
print(f"   Nodes: {graph_stats['n_nodes']}, Edges: {graph_stats['n_edges']}")
print(f"   Communities: {graph_stats['n_communities']}")
print(f"   Largest component: {graph_stats['largest_component_size']} artists "
      f"({graph_stats['largest_component_size']/graph_stats['n_nodes']*100:.0f}%)")
print(f"   Most connected: {nodes_by_id[graph_stats['most_connected_artist']]['name']}")
print(f"   Top connector:  {nodes_by_id[graph_stats['top_connector']]['name']}")

top_comms = comm_sizes.most_common(10)
print(f"\nTop 10 communities:")
for cid, size in top_comms:
    members = sorted([n for n in artists_data.values() if n["community"] == cid],
                     key=lambda n: n["degree"], reverse=True)
    print(f"   Comm {cid}: {size} artists — {', '.join(n['name'] for n in members[:4])}...")


📐 Building graph and computing metrics...
   Computing betweenness centrality...
   Running Louvain community detection...

📊 Graph summary:
   Nodes: 1884, Edges: 10929
   Communities: 354
   Largest component: 1535 artists (81%)
   Most connected: Nubya Garcia
   Top connector:  Kaidi Akinnibi

Top 10 communities:
   Comm 4: 222 artists — Dezron Douglas, PYJÆN, Waaju, Rob Luft...
   Comm 2: 210 artists — Damien Jurado, Andrew Bird, Iron & Wine, Kevin Morby...
   Comm 7: 168 artists — Yazmin Lacey, Jordan Rakei, Meshell Ndegeocello, SAULT...
   Comm 0: 149 artists — Nubya Garcia, Yussef Dayes, Moses Yoofee Trio, Oscar Jerome...
   Comm 10: 137 artists — Four Tet, Apparat, FKJ, Mocky...
   Comm 17: 113 artists — Avishai Cohen, Brad Mehldau, Joshua Redman, Aaron Parks...
   Comm 8: 110 artists — El Michels Affair, Menahan Street Band, The Olympians, Donny Hathaway...
   Comm 25: 89 artists — Sophie Hutchings, Niklas Paschburg, Dustin O'Halloran, Ólafur Arnalds...
   Comm 27: 78 artists 

In [12]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 7 — PREPARE VISUALISATION DATA + TRIBE LABELS
# ════════════════════════════════════════════════════════════════════════════

top_comm_ids = [c for c, _ in comm_sizes.most_common(TOP_N_COMMUNITIES)]

def auto_label(members):
    """Generate a simple label from top artist names (fallback if no API key)."""
    top_names = [m["name"] for m in members[:3]]
    return " / ".join(top_names)

def infer_tribe_label(artists, size):
    prompt = (
        f"Based on these {size} artists, give a 2-4 word genre/style label for this "
        f"musical cluster. Examples: 'British Neo-Jazz', 'Folk & Singer-Songwriter', "
        f"'Classic Jazz', 'Electronic & Ambient'. "
        f"Artists: {', '.join(artists[:10])}. "
        f"Respond with ONLY the label, nothing else."
    )
    result = anthropic_call(prompt, max_tokens=20)
    return result if result else auto_label(
        sorted([n for n in artists_data.values()
                if n["community"] == top_comm_ids[artists_data[list(artists_data.keys())[0]]["community"]]],
               key=lambda n: n["degree"], reverse=True))

print(f"🤖 Inferring tribe labels for {TOP_N_COMMUNITIES} communities...")
if not ANTHROPIC_API_KEY:
    print("   (No API key — using auto-generated labels from top artists)")

comm_labels = {}
nodes_list  = list(artists_data.values())

for cid in top_comm_ids:
    members = sorted([n for n in nodes_list if n.get("community") == cid],
                     key=lambda n: n["degree"], reverse=True)
    top_names = [n["name"] for n in members[:10]]

    if ANTHROPIC_API_KEY:
        label = infer_tribe_label(top_names, comm_sizes[cid])
        time.sleep(0.3)
    else:
        label = auto_label(members)

    comm_labels[cid] = label
    print(f"   Comm {cid} ({comm_sizes[cid]} artists) → {label}")

# ── Build viz_data ────────────────────────────────────────────────────────────
nodes_sorted = sorted(nodes_list, key=lambda x: (x["degree"], x["track_count"]), reverse=True)
top_ids      = set(n["id"] for n in nodes_sorted[:TOP_N_NODES])

viz_edges = [e for e in edges
             if e["source"] in top_ids and e["target"] in top_ids
             and e["weight"] >= MIN_EDGE_WEIGHT]

comm_members = {}
for cid in top_comm_ids:
    members = sorted([n for n in nodes_list if n.get("community") == cid],
                     key=lambda n: n["degree"], reverse=True)
    comm_members[str(cid)] = {
        "id":          cid,
        "size":        len(members),
        "label":       comm_labels[cid],
        "top_artists": [n["name"] for n in members[:15]],
        "all_artists": [n["name"] for n in members],
        "description": "",
    }

viz_data = {
    "nodes": [{"id": n["id"], "name": n["name"], "track_count": n["track_count"],
               "degree": n["degree"], "betweenness": n["betweenness"],
               "community": n.get("community", -1)}
              for n in nodes_sorted[:TOP_N_NODES]],
    "edges": viz_edges,
    "communities": comm_members,
    "discoveries": [],   # filled in next step
    "stats": {
        "total_liked_songs":  len(tracks),
        "total_artists":      len(artists_data),
        "total_edges":        len(edges),
        "most_connected":     nodes_by_id[graph_stats["most_connected_artist"]]["name"],
        "top_connector":      nodes_by_id[graph_stats["top_connector"]]["name"],
        "most_isolated":      nodes_by_id[graph_stats["most_isolated_artist"]]["name"],
        "n_components":       graph_stats["n_components"],
        "largest_component":  graph_stats["largest_component_size"],
        "n_communities":      graph_stats["n_communities"],
    },
}

print(f"\n✅ Viz data ready: {len(viz_data['nodes'])} nodes, {len(viz_data['edges'])} edges")


🤖 Inferring tribe labels for 10 communities...
   Comm 4 (222 artists) → Contemporary Jazz Fusion
   Comm 2 (210 artists) → Indie Folk & Americana
   Comm 7 (168 artists) → Neo-Soul & R&B
   Comm 0 (149 artists) → UK Jazz Fusion
   Comm 10 (137 artists) → Electronic & Jazz-Influenced
   Comm 17 (113 artists) → Contemporary Acoustic Jazz
   Comm 8 (110 artists) → Cinematic Soul Funk
   Comm 25 (89 artists) → Contemporary Neoclassical Piano
   Comm 27 (78 artists) → Modal & Post-Bop Jazz
   Comm 5 (74 artists) → Portuguese Singer-Songwriter

✅ Viz data ready: 1883 nodes, 4628 edges


In [13]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 8 — TRIBE DESCRIPTIONS + DISCOVERY CANDIDATES
# ════════════════════════════════════════════════════════════════════════════

# ── Part A: Tribe personality descriptions ────────────────────────────────────
print("🤖 Generating tribe descriptions...")
if not ANTHROPIC_API_KEY:
    print("   (No API key — using top artist names as description fallback)")

for cid in top_comm_ids:
    cm = comm_members[str(cid)]
    print(f"   {cm['label']}...", end=" ", flush=True)

    if ANTHROPIC_API_KEY:
        prompt = (
            f"You are a music personality analyst. Based on this cluster of {cm['size']} artists "
            f"from a listener's Spotify library, write a 2-3 sentence personality profile "
            f"revealing what this musical tribe says about the listener. Be specific and "
            f"evocative — mention 2-3 artist names, describe the sonic character, and say "
            f"something insightful about the personality behind this taste.\n\n"
            f"Top artists: {', '.join(cm['top_artists'][:12])}\n\n"
            f"Respond with ONLY the description, nothing else."
        )
        desc = anthropic_call(prompt, max_tokens=200)
        time.sleep(0.3)
    else:
        desc = None

    if not desc:
        desc = f"A collection of {cm['size']} artists including {', '.join(cm['top_artists'][:3])} and more."

    comm_members[str(cid)]["description"]              = desc
    viz_data["communities"][str(cid)]["description"]   = desc
    print("✅")

# ── Part B: Discovery candidates ─────────────────────────────────────────────
print("\n🔭 Ranking discovery candidates...")

# Precompute lookup: normalize_name(artist_name) → community_id
# (avoids O(candidates × recommenders × artists) nested loop)
_norm_name_to_community = {
    normalize_name(a["name"]): a.get("community", -1)
    for a in artists_data.values()
}

# Compute composite score for each candidate
# score = 40% coverage (how many of your artists recommend it)
#        + 30% mean similarity
#        + 30% max similarity
max_count = max((d["count"] for d in discovery_raw.values()), default=1)

candidates = []
for norm_name, d in discovery_raw.items():
    if d["count"] < 2:   # skip artists recommended by only one of yours
        continue
    score = (
        (d["count"] / max_count) * 0.4 +
        (d["score_sum"] / d["count"])   * 0.3 +
        d["max_score"]                  * 0.3
    )
    # Infer probable tribe from recommenders using precomputed lookup
    recommender_communities = [
        _norm_name_to_community[normalize_name(rec_name)]
        for rec_name in d["recommenders"]
        if normalize_name(rec_name) in _norm_name_to_community
    ]
    probable_tribe_id  = Counter(recommender_communities).most_common(1)[0][0] if recommender_communities else -1
    probable_tribe_lbl = comm_labels.get(probable_tribe_id, "Unknown")

    candidates.append({
        "name":              d["name"],
        "score":             round(score, 4),
        "recommender_count": d["count"],
        "mean_similarity":   round(d["score_sum"] / d["count"], 3),
        "max_similarity":    round(d["max_score"], 3),
        "recommenders":      d["recommenders"][:5],
        "probable_tribe":    probable_tribe_lbl,
        "probable_tribe_id": probable_tribe_id,
        "spotify_url":       f"https://open.spotify.com/search/{urlparse.quote(d['name'])}",
    })

candidates.sort(key=lambda x: x["score"], reverse=True)
viz_data["discoveries"] = candidates[:TOP_N_DISCOVERIES]

print(f"✅ {len(viz_data['discoveries'])} discovery candidates ranked.")
print(f"\nTop 10 artists you might like:")
for i, c in enumerate(viz_data["discoveries"][:10], 1):
    print(f"   {i:2}. {c['name']} (score: {c['score']:.3f}, "
          f"recommended by {c['recommender_count']} of your artists, "
          f"tribe: {c['probable_tribe']})")


🤖 Generating tribe descriptions...
   Contemporary Jazz Fusion... ✅
   Indie Folk & Americana... ✅
   Neo-Soul & R&B... ✅
   UK Jazz Fusion... ✅
   Electronic & Jazz-Influenced... ✅
   Contemporary Acoustic Jazz... ✅
   Cinematic Soul Funk... ✅
   Contemporary Neoclassical Piano... ✅
   Modal & Post-Bop Jazz... ✅
   Portuguese Singer-Songwriter... ✅

🔭 Ranking discovery candidates...
✅ 50 discovery candidates ranked.

Top 10 artists you might like:
    1. Joe Armon-Jones (score: 0.820, recommended by 83 of your artists, tribe: UK Jazz Fusion)
    2. Theo Croker (score: 0.680, recommended by 54 of your artists, tribe: UK Jazz Fusion)
    3. Greg Foat (score: 0.675, recommended by 55 of your artists, tribe: UK Jazz Fusion)
    4. Sonny Rollins (score: 0.654, recommended by 42 of your artists, tribe: Modal & Post-Bop Jazz)
    5. Tord Gustavsen Trio (score: 0.651, recommended by 46 of your artists, tribe: Contemporary Acoustic Jazz)
    6. Kamaal Williams (score: 0.647, recommended by 48 

In [14]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 9 — GENERATE HTML DASHBOARD & DOWNLOAD
# ════════════════════════════════════════════════════════════════════════════
import base64, os, json
from pathlib import Path

# ── Load HTML template ────────────────────────────────────────────────────────
# Priority:
#   1. Local file  sonic_atlas_template.html  (always used when running locally)
#   2. TEMPLATE_URL from .env / Colab Secrets (required when running in Colab)
# ─────────────────────────────────────────────────────────────────────────────
LOCAL_TEMPLATE = Path(__file__).parent / "sonic_atlas_template.html" \
    if "__file__" in dir() else Path("sonic_atlas_template.html")

if LOCAL_TEMPLATE.exists():
    print(f"📄 Loading template from local file: {LOCAL_TEMPLATE}")
    HTML_TEMPLATE = LOCAL_TEMPLATE.read_text(encoding="utf-8")
elif TEMPLATE_URL:
    print("📥 Downloading HTML template from URL...")
    r = requests.get(TEMPLATE_URL, timeout=15)
    if r.status_code != 200:
        raise RuntimeError(
            f"Failed to download template (HTTP {r.status_code}).\n"
            f"Check TEMPLATE_URL in your Colab Secrets / .env."
        )
    HTML_TEMPLATE = r.text
    print(f"   Template: {len(HTML_TEMPLATE)//1024} KB")
else:
    raise FileNotFoundError(
        "Template not found.\n"
        "  • Local: make sure sonic_atlas_template.html is in the sonic-atlas/ folder.\n"
        "  • Colab: set TEMPLATE_URL in the Secrets panel (🔑)."
    )

assert "%%DATA_BLOCK%%" in HTML_TEMPLATE, "Template is missing the %%DATA_BLOCK%% placeholder!"

# ── Inject data as base64 ─────────────────────────────────────────────────────
js_data    = json.dumps(viz_data, ensure_ascii=False)
data_b64   = base64.b64encode(js_data.encode("utf-8")).decode("ascii")
data_block = f'const _b64 = "{data_b64}";'

html_final = HTML_TEMPLATE.replace("%%DATA_BLOCK%%", data_block)

assert "%%DATA_BLOCK%%" not in html_final,    "Placeholder not replaced!"
assert "const GD = JSON.parse" in html_final, "GD variable not found in template!"

print(f"✅ HTML generated: {len(html_final)//1024} KB")
print(f"   Nodes: {len(viz_data['nodes'])}, Edges: {len(viz_data['edges'])}")
print(f"   Tribes: {[cm['label'] for cm in viz_data['communities'].values()]}")
print(f"   Discovery candidates: {len(viz_data['discoveries'])}")

# ── Save & download ───────────────────────────────────────────────────────────
output_path = "sonic_atlas.html"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(html_final)

try:
    from google.colab import files
    files.download(output_path)
    print("📥 Download started.")
except ImportError:
    print(f"ℹ️  File saved at: {os.path.abspath(output_path)}")


📄 Loading template from local file: sonic_atlas_template.html
✅ HTML generated: 960 KB
   Nodes: 1883, Edges: 4628
   Tribes: ['Contemporary Jazz Fusion', 'Indie Folk & Americana', 'Neo-Soul & R&B', 'UK Jazz Fusion', 'Electronic & Jazz-Influenced', 'Contemporary Acoustic Jazz', 'Cinematic Soul Funk', 'Contemporary Neoclassical Piano', 'Modal & Post-Bop Jazz', 'Portuguese Singer-Songwriter']
   Discovery candidates: 50
ℹ️  File saved at: /home/nuno_paiva_nos_pt/ml-ai-experiments/sonic-atlas/sonic_atlas.html


## 📡 Step 10 — Publish to GitHub Pages

> **One-time setup required before running the cell below.**
>
> The next cell will push `sonic_atlas.html` to the repo, but GitHub Pages must be enabled first so the file is publicly served.

### Enable GitHub Pages (do this once)

1. Go to your repo on GitHub: **https://github.com/nfpaiva/ml-ai-experiments**
2. Click **Settings** → **Pages** (left sidebar)
3. Under **Build and deployment**, set:
   - **Source**: `Deploy from a branch`
   - **Branch**: `main`  |  **Folder**: `/ (root)`
4. Click **Save**

After a minute, your atlas will be live at:
```
https://nfpaiva.github.io/ml-ai-experiments/sonic-atlas/sonic_atlas.html
```

Once Pages is enabled, run the cell below to push the latest HTML whenever you regenerate the atlas.


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 10 — PUSH HTML TO GITHUB
# ════════════════════════════════════════════════════════════════════════════
import subprocess, pathlib

# Resolve the absolute path of the generated HTML and repo root
html_path   = pathlib.Path(output_path).resolve()
repo_root   = html_path.parent.parent  # sonic-atlas/ -> repo root

# Path relative to repo root (used for git add)
html_rel    = html_path.relative_to(repo_root)

def _run(cmd, **kwargs):
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=repo_root, **kwargs)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}\n{result.stderr.strip()}")
    return result.stdout.strip()

print(f"📂 Repo root : {repo_root}")
print(f"📄 HTML file : {html_rel}")

# Force-add: sonic_atlas.html may be in .gitignore (intentionally excluded from
# regular commits, but we always want to push the latest dashboard build)
_run(["git", "add", "-f", str(html_rel)])
status = _run(["git", "status", "--short"])
print(f"   git status:\n{status or '   (nothing staged — file unchanged)'}")

if status:
    commit_msg = "chore: update sonic_atlas.html [auto]"
    # Disable GPG signing for this commit — the GPG agent is not available
    # in the Jupyter process even when it works in a regular terminal session.
    _run(["git", "-c", "commit.gpgsign=false", "commit", "-m", commit_msg])
    print(f"   Committed: {commit_msg}")
    _run(["git", "push"])
    print("✅ Pushed to origin/main")
    print()
    print("🌐 Enable GitHub Pages (if not done yet):")
    print("   Repo → Settings → Pages → Source: Deploy from a branch")
    print("   Branch: main  |  Folder: / (root)")
    if GITHUB_USERNAME and GITHUB_REPO:
        print(f"   URL will be: https://{GITHUB_USERNAME}.github.io/{GITHUB_REPO}/{html_rel}")
    else:
        print("   URL will be: https://<GITHUB_USERNAME>.github.io/<GITHUB_REPO>/" + str(html_rel))
        print("   ⚠️  Set GITHUB_USERNAME and GITHUB_REPO in .env to see your exact URL.")

else:
    print("ℹ️  HTML is already up-to-date in the repo — nothing to push.")


📂 Repo root : /home/nuno_paiva_nos_pt/ml-ai-experiments
📄 HTML file : sonic-atlas/sonic_atlas.html
   git status:
A  sonic-atlas/sonic_atlas.html
?? .gitignore
?? mlops-zoomcamp-experiment-tracking/src/excom-dummy-plots.ipynb
?? mlops-zoomcamp-experiment-tracking/src/test-distributions.ipynb
?? mlops-zoomcamp-experiment-tracking/src/test_sqlite_access.py
?? mlops-zoomcamp-experiment-tracking/test_sqlite_access.py
?? notebooks/dados_campanha_equipa_dedicada.csv
?? sonic-atlas/requirements.txt
?? sonic-atlas/sonic_atlas.ipynb
?? sonic-atlas/sonic_atlas_template.html


RuntimeError: Command failed: git commit -m chore: update sonic_atlas.html [auto]
error: gpg failed to sign the data
fatal: failed to write commit object

## 🚀 What's Next

You now have a `sonic_atlas.html` file — a self-contained interactive dashboard.

### Share it
- **GitHub Pages**: Push the HTML to a repo and enable Pages for a live URL
- **Netlify Drop**: Drag the file to [app.netlify.com/drop](https://app.netlify.com/drop) for instant hosting
- **LinkedIn Carousel**: Use the companion React carousel to create a visual post

### Extend it
- **Discovery panel**: The dashboard includes a "Discover" tab with artists you might like, ranked by how many of your artists recommend them
- **Re-run anytime**: Your library evolves — re-run the notebook every few months to see how your taste network changes
- **Other sources**: Adapt the pipeline to use a playlist URL instead of Liked Songs

### Technical notes
- The HTML file is self-contained (no server needed) — all data is embedded as base64
- The template is fetched from GitHub Gist — update the template and regenerate to get changes
- Community detection uses the Louvain algorithm with `random_state=42` for reproducibility
